In [1]:
import pandas as pdimport numpy as npimport jsonfrom datetime import datetime# Set random seed for reproducibilitynp.random.seed(42)# Generate mock dataset for causal analysisn_samples = 1000# Generate confoundersage = np.random.normal(45, 12, n_samples).astype(int)age = np.clip(age, 18, 80)gender = np.random.binomial(1, 0.5, n_samples)  # 0: Female, 1: Maleeducation_years = np.random.normal(14, 3, n_samples).astype(int)education_years = np.clip(education_years, 8, 20)# Socioeconomic status (1: Low, 2: Medium, 3: High)ses_probs = [0.3, 0.5, 0.2]ses = np.random.choice([1, 2, 3], n_samples, p=ses_probs)# Generate treatment assignment with confounding# Higher education and SES increase probability of treatmenttreatment_prob = 1 / (1 + np.exp(-(-2 + 0.1*education_years + 0.5*ses + 0.02*age)))treatment = np.random.binomial(1, treatment_prob, n_samples)# Generate outcome with treatment effect and confounding# True treatment effect = 2.5base_outcome = 10 + 0.05*age + 0.3*education_years + ses + np.random.normal(0, 2, n_samples)outcome = base_outcome + 2.5*treatment + np.random.normal(0, 1, n_samples)# Generate additional covariatesbmi = np.random.normal(25, 4, n_samples)bmi = np.clip(bmi, 15, 45)physical_activity = np.random.poisson(3, n_samples)  # days per weekphysical_activity = np.clip(physical_activity, 0, 7)comorbidity_count = np.random.poisson(1, n_samples)comorbidity_count = np.clip(comorbidity_count, 0, 5)# Create labels for categorical variablesgender_labels = {0: 'Female', 1: 'Male'}ses_labels = {1: 'Low', 2: 'Medium', 3: 'High'}# Create DataFramedata = pd.DataFrame({    'participant_id': range(1, n_samples + 1),    'age': age,    'gender': gender,    'education_years': education_years,    'ses': ses,    'treatment': treatment,    'outcome': outcome,    'bmi': bmi,    'physical_activity': physical_activity,    'comorbidity_count': comorbidity_count,    'gender_label': [gender_labels[g] for g in gender],    'ses_label': [ses_labels[s] for s in ses]})# Save datasetdata.to_csv('mock_causal_dataset.csv', index=False)# Create data dictionarydata_dict = pd.DataFrame({    'Variable': ['participant_id', 'age', 'gender', 'education_years', 'ses', 'treatment',                  'outcome', 'bmi', 'physical_activity', 'comorbidity_count', 'gender_label', 'ses_label'],    'Type': ['Integer', 'Integer (years)', 'Binary (0/1)', 'Integer (years)', 'Categorical (1-3)',              'Binary (0/1)', 'Continuous', 'Continuous', 'Integer (0-7)', 'Integer (0-5)',              'Categorical', 'Categorical'],    'Description': ['Unique identifier for each participant', 'Age in years',                     'Gender (0: Female, 1: Male)', 'Years of education completed',                     'Socioeconomic status (1: Low, 2: Medium, 3: High)',                     'Treatment indicator (0: Control, 1: Treated)',                     'Continuous outcome measure', 'Body Mass Index',                     'Physical activity days per week', 'Number of comorbidities',                     'Gender label', 'SES label'],    'Range': ['1-1000', '18-80', '0 or 1', '8-20', '1-3', '0 or 1', 'Continuous',               '15-45', '0-7', '0-5', 'Female/Male', 'Low/Medium/High']})data_dict.to_csv('mock_data_dictionary.csv', index=False)# Generate summary statisticssummary_stats = {    'dataset_info': {        'n_samples': n_samples,        'n_features': len(data.columns),        'treatment_prevalence': float(treatment.mean()),        'outcome_mean': float(outcome.mean()),        'outcome_std': float(outcome.std())    },    'treatment_effect': {        'true_effect': 2.5,        'observed_difference': float(data[data['treatment']==1]['outcome'].mean() - data[data['treatment']==0]['outcome'].mean())    },    'confounding_summary': {        'age_treated_mean': float(data[data['treatment']==1]['age'].mean()),        'age_control_mean': float(data[data['treatment']==0]['age'].mean()),        'education_treated_mean': float(data[data['treatment']==1]['education_years'].mean()),        'education_control_mean': float(data[data['treatment']==0]['education_years'].mean()),        'ses_treated_dist': data[data['treatment']==1]['ses'].value_counts().to_dict(),        'ses_control_dist': data[data['treatment']==0]['ses'].value_counts().to_dict()    }}# Save summary statisticswith open('mock_dataset_summary.json', 'w') as f:    json.dump(summary_stats, f, indent=2)print("Mock dataset generated successfully!")print(f"Dataset shape: {data.shape}")print(f"Treatment prevalence: {treatment.mean():.2%}")print(f"True treatment effect: 2.5")print(f"Observed difference: {summary_stats['treatment_effect']['observed_difference']:.2f}")print("\nFirst few rows:")print(data.head())

Mock dataset generated successfully!Dataset shape: (1000, 12)Treatment prevalence: 74.20%True treatment effect: 2.5Observed difference: 3.15First few rows:   participant_id  age  gender  education_years  ses  treatment    outcome  \0               1   50       0               13    1          1  17.728677   1               2   43       0               11    2          1  21.784707   2               3   52       1               14    2          1  22.499295   3               4   63       1               18    1          1  19.939871   4               5   42       0                8    1          1  17.727841            bmi  physical_activity  comorbidity_count gender_label ses_label  0  26.430645                  2                  0       Female       Low  1  25.258024                  5                  1       Female    Medium  2  28.626177                  3                  0         Male    Medium  3  25.289008                  5                  0         Male       Low  4  33.23

In [2]:
# Let me check the current working directory and list the files we createdimport osprint("Current working directory:", os.getcwd())print("\nFiles in current directory:")for file in os.listdir('.'):    if file.startswith('mock_'):        print(f"- {file}")# Read back the dataset to verify it was created correctlyif os.path.exists('mock_causal_dataset.csv'):    df = pd.read_csv('mock_causal_dataset.csv')    print(f"\nDataset loaded successfully: {df.shape}")    print("\nColumn names:", list(df.columns))    print("\nData types:")    print(df.dtypes)    print("\nSummary statistics:")    print(df.describe())

Current working directory: /Users/vishal/Projects/opencausol/packages/opencodeFiles in current directory:- mock_dataset_summary.json- mock_data_dictionary.csv- mock_causal_dataset.csvDataset loaded successfully: (1000, 12)Column names: ['participant_id', 'age', 'gender', 'education_years', 'ses', 'treatment', 'outcome', 'bmi', 'physical_activity', 'comorbidity_count', 'gender_label', 'ses_label']Data types:participant_id         int64age                    int64gender                 int64education_years        int64ses                    int64treatment              int64outcome              float64bmi                  float64physical_activity      int64comorbidity_count      int64gender_label          objectses_label             objectdtype: objectSummary statistics:       participant_id          age       gender  education_years          ses  \count     1000.000000  1000.000000  1000.000000      1000.000000  1000.000000   mean       500.500000    44.765000     0.515000        13.5770